# Fig 5b: overall survival of TIL-high versus TIL-low patients

**Produces**
- **Fig 5b**: Kaplan–Meier curves and RMST (τ = 3 years) of TIL-high vs TIL-low patients, adjusted
  for MGMT: n = 207 (54 TIL-high, 153 TIL-low), +3.93 months [0.5, 7.4], p = 0.0251.
- TIL⁺ fraction versus age in the Fig 5b sections (Spearman ρ = −0.22, p = 0.001) and by MGMT status.
- A sensitivity table: the Fig 5b model with MGMT + age (+ sex, + p53), without the probe-batch
  blocks, and on the corrected cohort (every section mapped to its block, 260-patient survival
  cohort, complete cases: n = 212, +4.00 months [0.6, 7.4], p = 0.022).

The model: TIL⁺ fraction per Visium section (sections with ≥ 100 spots, all 228 qualify), TIL-high =
top quartile (cut-off 0.0304); sections without TIL⁺ spots are TIL-low. A section links to the
clinical tables by removing `_vis` and then the last letter of its name, matched to the first nine
characters of the subject ID, so the 21 backup sections (`_vbu`, `_vb2`, …) find no subject. Every
row of the clinical tables is eligible (first row per subject), so 10 Charité patients
outside the 260-patient survival cohort (immature follow-up) enter. A missing MGMT result counts as
unmethylated. The cell prints how many sections and patients each of these affects.

**Origin**: Spencer Watson, TIL_Niche_Comparisons (EFS `Analysis_Repo/Notebooks/`), cell "RMST Survival
Analysis" (as reproduced in `analysis/scratch/review_2026-11/source_cells_spencer_2026-11.ipynb`, cells 16–17).

**Compared with the published panel**: reproduced (n = 207, 54 / 153, +3.9 months, p = 0.0251).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lifelines import KaplanMeierFitter
from scipy.stats import spearmanr

from gbmcollaboration import PROBE_ISSUE_CHUV, PROBE_ISSUE_UKER, AllDataCenters
from gbmcollaboration.paper import cohort, config, output, til  # noqa: F401  (config sets R_HOME)
from gbmcollaboration.paper.ids import parse_ids

import rpy2.robjects as ro  # noqa: E402
from rpy2.robjects import pandas2ri  # noqa: E402
from rpy2.robjects.conversion import localconverter  # noqa: E402

ro.r("suppressPackageStartupMessages({library(survival); library(survRM2)})")

In [ ]:
# Parameters
OUT = output.setup("fig5b_TIL_survival")
QUANTILE_CUTOFF = 0.75
MIN_TOTAL_SPOTS = 100
TAU_YEARS = 3.0
TIME_COL, EVENT_COL = "os_years", "os_event"
CLINICAL_ID_COL = "Subject Id"

# A missing MGMT, sex or p53 value counts as unmethylated, male or negative; a missing age drops the patient.
COVARIATE_SETS = {
    "MGMT (published)": ["mgmt"],
    "MGMT + age": ["mgmt", "age"],
    "MGMT + age + sex": ["mgmt", "age", "female"],
    "MGMT + age + sex + p53": ["mgmt", "age", "female", "p53_pos"],
}

# Figure
FIGSIZE = (9, 6)
COLOR_HIGH, COLOR_LOW = "#fe7a02", "#007ebc"
TITLE_FONTSIZE, AXIS_LABEL_FONTSIZE, TICK_FONTSIZE = 8, 12, 12
LEGEND_FONTSIZE, ANNOT_FONTSIZE = 20, 12
KM_LINEWIDTH, CENSOR_MARKERSIZE, CENSOR_MARKEREDGEWIDTH = 2, 6, 1.5

In [ ]:
def to_r(name: str, df: pd.DataFrame) -> None:
    with localconverter(ro.default_converter + pandas2ri.converter):
        ro.globalenv[name] = ro.conversion.py2rpy(df.reset_index(drop=True))


def rmst(df: pd.DataFrame, arm: str, covariates=(), tau: float = 3.0,
         time: str = "os_years", event: str = "os_event") -> dict:
    """survRM2::rmst2 of arm 1 vs arm 0 at tau, adjusted for covariates if given.

    Rows with a missing value in any used column are dropped (complete cases), so code
    covariates beforehand if missing values should count as a level.
    Returns n, n_high (arm 1), n_low, rmst_high, rmst_low, diff_unadj, p_unadj and the
    (adjusted) diff, lo, hi, p, all in months.
    """
    cols = [time, event, arm, *covariates]
    d = df[cols].dropna()
    d = d[d[time] > 0]
    r = d.copy()
    r.columns = ["time", "event", "arm", *[f"c{i}" for i in range(len(covariates))]]
    to_r("sd", r)
    ro.globalenv["tv"] = float(tau)
    ro.r("a <- rmst2(sd$time, sd$event, sd$arm, tau = tv)")
    out = dict(n=len(d), n_high=int(d[arm].sum()), n_low=int((d[arm] == 0).sum()),
               rmst_high=ro.r("a$RMST.arm1$rmst[['Est.']]")[0] * 12,
               rmst_low=ro.r("a$RMST.arm0$rmst[['Est.']]")[0] * 12)
    u = list(ro.r("a$unadjusted.result[1,]"))
    out.update(diff_unadj=u[0] * 12, p_unadj=u[3])
    if covariates:
        cs = ", ".join(f"sd$c{i}" for i in range(len(covariates)))
        a = list(ro.r(f"rmst2(sd$time, sd$event, sd$arm, tau = tv, covariates = cbind({cs}))$adjusted.result[1,]"))
        out.update(diff=a[0] * 12, lo=a[1] * 12, hi=a[2] * 12, p=a[3])
    else:
        out.update(diff=u[0] * 12, lo=u[1] * 12, hi=u[2] * 12, p=u[3])
    return out

## Inputs

The spot-level TIL⁺ calls (frozen zone file) and every row of the three processed
clinical tables.

In [ ]:
spots = til.til_zones()
clin_out = AllDataCenters.load_clinical(exclude_idh1_mut=False)
print(f"{len(spots):,} spots in {spots['sample'].nunique()} sections, {int(spots['TIL_positive'].sum()):,} TIL+ spots")
print(f"Clinical rows: {len(clin_out)} ({clin_out['center'].value_counts().to_dict()})")

## Fig 5b, as published

TIL fraction per section, top-quartile split, clinical join and MGMT coding. RMST at τ = 3 years from survRM2, unadjusted and adjusted for MGMT.

In [ ]:
df_til = (spots.assign(_til=spots["TIL_positive"].astype(int))
          .groupby("sample", as_index=False)
          .agg(TIL_spots=("_til", "sum"), Total_Spots=("_til", "size")))
df_til["TIL_fraction"] = df_til["TIL_spots"] / df_til["Total_Spots"]
df_til = df_til[df_til["Total_Spots"] >= MIN_TOTAL_SPOTS].copy()
cutoff = df_til["TIL_fraction"].quantile(QUANTILE_CUTOFF)
df_til["TIL_group"] = np.where(df_til["TIL_fraction"] >= cutoff, "TIL_high", "TIL_low")
print(f"TIL fraction cutoff (Q={QUANTILE_CUTOFF}): {cutoff:.4f}; "
      f"{df_til['TIL_group'].value_counts().to_dict()} over {len(df_til)} sections")

# Clinical linkage
df_til["join_key"] = (df_til["sample"].astype(str).str.replace(r"_vis$", "", regex=True)
                      .str.replace(r"[a-z]$", "", regex=True))
clin_work = clin_out.copy()
clin_work["join_key"] = clin_work[CLINICAL_ID_COL].astype(str).str[:9]
clin_dedup = clin_work.drop_duplicates(subset=["join_key"])
clin_dedup = cohort.code_covariates(clin_dedup, missing="as_reference")
clin_os = clin_dedup.dropna(subset=[TIME_COL, EVENT_COL])

df = df_til.merge(clin_os[["join_key", TIME_COL, EVENT_COL, "mgmt", "age", "female", "p53_pos",
                           "MGMT promoter methylation"]], on="join_key", how="inner")
df = df[df[TIME_COL] > 0].dropna(subset=[TIME_COL, EVENT_COL]).copy()
df["arm"] = (df["TIL_group"] == "TIL_high").astype(int)

# What the join and the coding did
unjoined = df_til[~df_til["join_key"].isin(clin_os["join_key"])]
backup = unjoined["sample"].str.contains(r"_(?:vbu|vb2|bu2|bu3)$")
print(f"\nAfter clinical merge: {len(df)} patients, TIL-high {int(df['arm'].sum())}, "
      f"TIL-low {int((df['arm'] == 0).sum())}, events {int(df[EVENT_COL].sum())}")
print(f"Sections without a clinical match: {len(unjoined)} ({int(backup.sum())} backup sections, "
      f"{int((~backup).sum())} others: {unjoined.loc[~backup, 'sample'].tolist()})")
print(f"Missing MGMT counted as unmethylated: {int(df['MGMT promoter methylation'].isna().sum())}")
n_zero = int((df["TIL_spots"] == 0).sum())
n_zero_mgmt = int(((df["TIL_spots"] == 0) & df["MGMT promoter methylation"].notna()).sum())
print(f"Sections without TIL+ spots in the model (all TIL-low): {n_zero}; with an MGMT result: {n_zero_mgmt}")
outside = sorted(set(df["join_key"]) - set(cohort.survival_cohort()["patient"]))
print(f"Fitted patients outside the 260-patient survival cohort: {len(outside)} {outside}")

published = rmst(df, "arm", ["mgmt"], tau=TAU_YEARS)
print(f"\nRMST TIL-high {published['rmst_high']:.1f} mo, TIL-low {published['rmst_low']:.1f} mo")
print(f"Unadjusted: {published['diff_unadj']:+.2f} mo, p = {published['p_unadj']:.4f}")
print(f"MGMT-adjusted: {published['diff']:+.2f} mo [{published['lo']:.1f}, {published['hi']:.1f}], "
      f"p = {published['p']:.4f} (published +3.9, p = 0.0251)")

In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE)
for grp, color, label in (("TIL_high", COLOR_HIGH, "TIL-High"), ("TIL_low", COLOR_LOW, "TIL-Low")):
    mask = df["TIL_group"] == grp
    kmf = KaplanMeierFitter().fit(df.loc[mask, TIME_COL], df.loc[mask, EVENT_COL], label=label)
    kmf.plot_survival_function(ax=ax, color=color, linewidth=KM_LINEWIDTH, ci_show=True, ci_alpha=0.2)
    for t in df.loc[mask & (df[EVENT_COL] == 0), TIME_COL]:
        ax.plot(t, kmf.predict(t), "|", color=color, markersize=CENSOR_MARKERSIZE,
                markeredgewidth=CENSOR_MARKEREDGEWIDTH)
ax.axvline(TAU_YEARS, color="gray", linestyle="--", linewidth=0.8, alpha=0.7, label=f"τ={TAU_YEARS:g}yr")
ax.text(0.98, 0.98,
        f"RMST (τ={TAU_YEARS:g}yr):\n  High: {published['rmst_high'] / 12:.2f} yrs\n"
        f"  Low: {published['rmst_low'] / 12:.2f} yrs\n  Δ: {published['diff']:+.1f} months\n"
        f"  p(adj) = {published['p']:.4f}\n  p(unadj) = {published['p_unadj']:.4f}",
        transform=ax.transAxes, fontsize=ANNOT_FONTSIZE, va="top", ha="right",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
ax.set_title(f"Overall Survival: TIL-High (n={published['n_high']}) vs TIL-Low (n={published['n_low']})",
             fontsize=TITLE_FONTSIZE, fontweight="bold")
ax.set_xlabel("Time (years)", fontsize=AXIS_LABEL_FONTSIZE)
ax.set_ylabel("Survival probability", fontsize=AXIS_LABEL_FONTSIZE)
ax.tick_params(axis="both", labelsize=TICK_FONTSIZE)
ax.set_ylim(0, 1.05)
ax.set_xlim(0, None)
ax.legend(loc="lower left", frameon=False, fontsize=LEGEND_FONTSIZE)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
output.save(OUT, "fig5b_til_km_rmst", fig, pd.DataFrame([published]), panel="Fig 5b")

## TIL⁺ fraction and age

Spearman correlation of TIL⁺ fraction with age at diagnosis in the Fig 5b sections, and
TIL⁺ fraction by MGMT status.

In [ ]:
rho, p_rho = spearmanr(df["TIL_fraction"], df["age"], nan_policy="omit")
print(f"TIL+ fraction vs age, Fig 5b sections (n = {int(df['age'].notna().sum())}): "
      f"Spearman rho = {rho:.2f}, p = {p_rho:.3g}")
for status, g in df.groupby("MGMT promoter methylation", dropna=False):
    print(f"  MGMT {status}: n = {len(g)}, median TIL+ fraction {g['TIL_fraction'].median():.4f}")

## Sensitivity table

The Fig 5b model (same sections, join and coding) with other covariate sets, then the
same model without the blocks of the probe batch flagged at QC, then the corrected cohort: every section mapped to its block (backup sections included),
the paper's 260-patient survival cohort, missing covariates left missing (complete
cases), cut-off over all 228 sections.

In [ ]:
rows = [dict(analysis="Fig 5b model", covariates=k, **rmst(df, "arm", v, tau=TAU_YEARS))
        for k, v in COVARIATE_SETS.items()]

probe_blocks = set(parse_ids(list(PROBE_ISSUE_CHUV) + list(PROBE_ISSUE_UKER))["block"])
df_probe = df[~parse_ids(df["sample"])["block"].isin(probe_blocks).to_numpy()]
print(f"Probe-batch sections in the Fig 5b model: {len(df) - len(df_probe)}")
rows += [dict(analysis="Fig 5b model, without probe-batch blocks", covariates=k,
              **rmst(df_probe, "arm", v, tau=TAU_YEARS))
         for k, v in COVARIATE_SETS.items() if k in ("MGMT (published)", "MGMT + age + sex + p53")]

groups = til.til_groups()
corrected = (groups.reset_index(names="sample")
             .merge(cohort.code_covariates(cohort.survival_cohort(), missing="keep"),
                    left_on="block", right_index=True))
corrected["arm"] = (corrected["TIL_group"] == "TIL_high").astype(int)
rows += [dict(analysis="Corrected cohort (all sections mapped, survival cohort, complete cases)",
              covariates=k, **rmst(corrected, "arm", v, tau=TAU_YEARS))
         for k, v in {"none": [], **COVARIATE_SETS}.items()]
rho_c, p_c = spearmanr(corrected["TIL_fraction"], corrected["age"], nan_policy="omit")
print(f"TIL+ fraction vs age, corrected cohort: rho = {rho_c:.2f}, p = {p_c:.3g}")

sens = pd.DataFrame(rows)
cols = ["analysis", "covariates", "n", "n_high", "n_low", "diff", "lo", "hi", "p", "diff_unadj", "p_unadj"]
output.save_table(OUT, "fig5b_sensitivity", sens[cols].round(4), index=False)
print(sens[cols].round({"diff": 2, "lo": 1, "hi": 1, "p": 4, "diff_unadj": 2, "p_unadj": 4}).to_string(index=False))

In [ ]:
print(output.versions(["numpy", "pandas", "scipy", "lifelines", "matplotlib", "rpy2"], ["survival", "survRM2"])
      .to_string(index=False))